In [1]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 3060 Ti


In [3]:
DATASET_DIR = Path("dataset/CUB_200_2011")
IMAGES_DIR = DATASET_DIR / "images"

IMAGES_FILE = DATASET_DIR / "images.txt"
LABELS_FILE = DATASET_DIR / "image_class_labels.txt"
SPLIT_FILE = DATASET_DIR / "train_test_split.txt"
CLASSES_FILE = DATASET_DIR / "classes.txt"  


required_paths = [IMAGES_DIR, IMAGES_FILE, LABELS_FILE, SPLIT_FILE]
missing = [str(path) for path in required_paths if not path.exists()]

if missing:
    raise FileNotFoundError(
        "Nu am găsit următoarele fișiere/foldere: "
        + ", ".join(missing)
        + "\nVerifică DATASET_DIR și structura datasetului."
    )

print("Dataset found:", DATASET_DIR.resolve())

Dataset found: C:\Users\andre\Documents\GitHub\Project-CV3\dataset\CUB_200_2011


In [4]:
# ID + value
images_df = pd.read_csv(
    IMAGES_FILE, sep=" ", header=None, names=["image_id", "filepath"]
)
labels_df = pd.read_csv(
    LABELS_FILE, sep=" ", header=None, names=["image_id", "class_id"]
)
split_df = pd.read_csv(
    SPLIT_FILE, sep=" ", header=None, names=["image_id", "is_train"]
)

# Combinăm informațiile după ID-ul imaginii.
metadata = images_df.merge(labels_df, on="image_id").merge(
    split_df, on="image_id"
)

# 200 classes
metadata["label"] = metadata["class_id"] - 1
metadata["image_path"] = metadata["filepath"].apply(
    lambda p: IMAGES_DIR / p
)


print("Total images:", len(metadata))
print("Total classes:", metadata["label"].nunique())
print("Official split (will not be used):")
print("Train images:", int((metadata["is_train"] == 1).sum()))
print("Test images:", int((metadata["is_train"] == 0).sum()))
metadata.head()

Total images: 11788
Total classes: 200
Official split (will not be used):
Train images: 5994
Test images: 5794


,image_id,filepath,class_id,is_train,label,image_path
0,1,001.Black_footed_Albatross/Black_Footed_Albatr...,1,0,0,dataset\CUB_200_2011\images\001.Black_footed_A...
1,2,001.Black_footed_Albatross/Black_Footed_Albatr...,1,1,0,dataset\CUB_200_2011\images\001.Black_footed_A...
2,3,001.Black_footed_Albatross/Black_Footed_Albatr...,1,0,0,dataset\CUB_200_2011\images\001.Black_footed_A...
3,4,001.Black_footed_Albatross/Black_Footed_Albatr...,1,1,0,dataset\CUB_200_2011\images\001.Black_footed_A...
4,5,001.Black_footed_Albatross/Black_Footed_Albatr...,1,1,0,dataset\CUB_200_2011\images\001.Black_footed_A...


In [5]:
from sklearn.model_selection import train_test_split

SEED = 42

train_df, temp_df = train_test_split(
    metadata,
    test_size=0.20,
    random_state=SEED,
    stratify=metadata["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 9430
Validation: 1179
Test: 1179


In [6]:
from torchvision import transforms

IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [7]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class CUB200Dataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image = Image.open(row["image_path"]).convert("RGB")
        label = int(row["label"])

        if self.transform:
            image = self.transform(image)

        return image, label

In [8]:
train_dataset = CUB200Dataset(train_df, train_transform)
val_dataset = CUB200Dataset(val_df, eval_transform)
test_dataset = CUB200Dataset(test_df, eval_transform)

BATCH_SIZE = 32
NUM_WORKERS = 0

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

In [9]:
images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)

Images shape: torch.Size([32, 3, 224, 224])
Labels shape: torch.Size([32])
